# Topic 20 — Reasoning models and test-time compute

Thinking tokens are generated before (and, with interleaved thinking, between) the
visible output. They are not returned as text, they do occupy the context window,
and they are **billed as output tokens**.

This notebook runs the hand-written `reasoning.py` module: billing math, the
effort ladder, the `max_output_tokens` trap, parallel vs serial test-time compute,
`choose_effort`, reasoning-model prompting, and planner/worker routing.

Everything is offline and deterministic.

In [1]:
import sys

sys.path.insert(0, '.')  # run from the implementation/ folder

from reasoning import (
    BASE_LATENCY_S, EFFORT_ORDER, EFFORT_TO_THINKING_BUDGET, PRICE_TABLE,
    RECOMMENDED_REASONING_RESERVE, THINKING_TOKENS_PER_SECOND, VISIBLE_TOKENS_PER_SECOND,
    Difficulty, Effort, ModelSpec, OutcomeVerifier, ProcessVerifier, Sample, SubTask,
    TaskProfile, Usage, best_of_n, build_reasoning_prompt, choose_effort, cost_per_request,
    detect_and_strip_step_by_step, detect_step_by_step, estimate, estimate_cost,
    majority_vote, parallel_cost, planned_thinking_tokens, planner_worker_split,
    recommended_reserve_warnings, refine, route_turn, run_completion, serial_cost,
    validate_thinking_budget, will_fit,
)

SMALL = PRICE_TABLE['reasoning-small']
print('module loaded; reasoning-small prices =', SMALL)

module loaded; reasoning-small prices = Price(input_per_mtok=0.55, output_per_mtok=2.2, cached_input_per_mtok=0.055)


## 1. Thinking tokens are billed as OUTPUT tokens

`billed_output = visible_output + reasoning`, both at the output rate. The input
price is never applied to thinking.

In [2]:
usage = Usage(input_tokens=800, visible_output_tokens=300, reasoning_tokens=4096)
bill = estimate_cost(usage, SMALL)

print(f'billed output tokens = {usage.billed_output_tokens}  (visible {usage.visible_output_tokens} + thinking {usage.reasoning_tokens})')
print(f'input cost           = ${bill.input_cost:.6f}')
print(f'visible output cost  = ${bill.visible_output_cost:.6f}')
print(f'thinking cost        = ${bill.reasoning_cost:.6f}')
print(f'total                = ${bill.total_cost:.6f}')
print(f'thinking share of the bill = {bill.reasoning_share:.1%}')

without_thinking = cost_per_request(800, 300, 0, SMALL)
print(f'same answer, no thinking    = ${without_thinking:.6f}  -> thinking is {bill.total_cost / without_thinking:.1f}x the rest of the call')

billed output tokens = 4396  (visible 300 + thinking 4096)
input cost           = $0.000440
visible output cost  = $0.000660
thinking cost        = $0.009011
total                = $0.010111
thinking share of the bill = 89.1%
same answer, no thinking    = $0.001100  -> thinking is 9.2x the rest of the call


### The provider usage fields

OpenAI exposes `usage.output_tokens_details.reasoning_tokens`; Anthropic exposes
`usage.output_tokens_details.thinking_tokens`, and when streaming that count only
lands on the **final `message_delta`**.

In [3]:
print('openai   :', usage.to_usage_payload('openai'))
print('anthropic (non-streaming):', usage.to_usage_payload('anthropic'))
print('anthropic (early delta)  :', usage.to_usage_payload('anthropic', streaming=True, final_delta=False))
print('anthropic (final delta)  :', usage.to_usage_payload('anthropic', streaming=True, final_delta=True))

openai   : {'input_tokens': 800, 'output_tokens': 4396, 'output_tokens_details': {'reasoning_tokens': 4096}}
anthropic (non-streaming): {'input_tokens': 800, 'output_tokens': 4396, 'output_tokens_details': {'thinking_tokens': 4096}}
anthropic (early delta)  : {'input_tokens': 800, 'output_tokens': 4396, 'output_tokens_details': {'thinking_tokens': None}}
anthropic (final delta)  : {'input_tokens': 800, 'output_tokens': 4396, 'output_tokens_details': {'thinking_tokens': 4096}}


## 2. The knobs: an effort ladder and a thinking budget

`EFFORT_TO_THINKING_BUDGET` is the one-number summary of adaptive thinking; the
full picture is per (effort, difficulty), and `low` effort may skip thinking
entirely on easy input.

In [4]:
print('adaptive target thinking tokens by effort x difficulty')
print(f"{'effort':8s} {'easy':>8s} {'medium':>8s} {'hard':>8s}")
for effort in EFFORT_ORDER:
    row = [planned_thinking_tokens(effort, d) for d in Difficulty]
    print(f'{effort.value:8s} {row[0]:8d} {row[1]:8d} {row[2]:8d}')

print()
print('low effort on an easy prompt thinks', planned_thinking_tokens(Effort.LOW, Difficulty.EASY), 'tokens')
print('low effort on a hard prompt thinks ', planned_thinking_tokens(Effort.LOW, Difficulty.HARD), 'tokens')

adaptive target thinking tokens by effort x difficulty
effort       easy   medium     hard
none            0        0        0
minimal         0      128      512
low             0     1024     2048
medium       1024     4096     8192
high         4096    16384    32768
xhigh        8192    32768    65536
max         16384    65536   131072

low effort on an easy prompt thinks 0 tokens
low effort on a hard prompt thinks  2048 tokens


### Manual extended-thinking budgets are validated

Min 1024, and strictly less than `max_tokens` — except with interleaved thinking,
whose budget spans every thinking block in one turn.

In [5]:
for budget, max_tokens, interleaved in [(2048, 4096, False), (1024, 4096, False), (2048, 2048, False), (4096, 2048, True), (512, 4096, False)]:
    label = f'budget={budget} max_tokens={max_tokens} interleaved={interleaved}'
    try:
        validate_thinking_budget(budget, max_tokens, interleaved=interleaved)
        print(f'OK    {label}')
    except (ValueError, TypeError) as exc:
        print(f'ERROR {label} -> {exc}')

OK    budget=2048 max_tokens=4096 interleaved=False
OK    budget=1024 max_tokens=4096 interleaved=False
ERROR budget=2048 max_tokens=2048 interleaved=False -> manual extended thinking requires budget_tokens < max_tokens (got budget=2048, max_tokens=2048); interleaved thinking is the exception because its budget spans every thinking block in the turn
OK    budget=4096 max_tokens=2048 interleaved=True
ERROR budget=512 max_tokens=4096 interleaved=False -> thinking budget_tokens must be >= 1024 (got 512)


## 3. The `max_output_tokens` trap

`max_output_tokens` caps thinking + output **together**. Thinking is spent first,
so a call can run out of budget before the first visible token — and still bill
input + reasoning.

OpenAI advises reserving roughly 25,000 tokens for reasoning + output while
experimenting.

In [6]:
result = run_completion(
    prompt='prove the recurrence',
    input_tokens=1000,
    planned_thinking=30_000,
    planned_visible=500,
    max_output_tokens=20_000,
    price=SMALL,
)
print('status           :', result.status)
print('incomplete_reason:', result.incomplete_reason)
print('visible text     :', repr(result.visible_text))
print('billed output    :', result.usage.billed_output_tokens, 'tokens (thinking', result.usage.reasoning_tokens, '+ visible', result.usage.visible_output_tokens, ')')
print(f'cost charged     : ${result.cost:.5f}   <-- paid, got nothing visible')
print('summary available:', result.thought_summary)

print()
print('will_fit(20000, 20000 planned thinking, 500 visible) =', will_fit(20_000, 20_000, 500))
print('will_fit(20000, 19000 planned thinking, 500 visible) =', will_fit(20_000, 19_000, 500))
for warning in recommended_reserve_warnings(8_000):
    print('WARN:', warning)
print(f'no warning at the {RECOMMENDED_REASONING_RESERVE:,}-token reserve:', recommended_reserve_warnings(RECOMMENDED_REASONING_RESERVE) == [])

status           : incomplete
incomplete_reason: max_output_tokens
visible text     : ''
billed output    : 20000 tokens (thinking 20000 + visible 0 )
cost charged     : $0.04455   <-- paid, got nothing visible
summary available: (readable summary of 20000 hidden thinking tokens)

will_fit(20000, 20000 planned thinking, 500 visible) = False
will_fit(20000, 19000 planned thinking, 500 visible) = True
WARN: max_output_tokens=8000 is below the ~25,000 tokens worth reserving for reasoning + output while experimenting; raising it and lowering effort is safer than getting status=incomplete
no warning at the 25,000-token reserve: True


## 4. Test-time compute: parallel and serial

* **Parallel** — draw N samples (best-of-N) or vote (self-consistency). Latency ≈
  one sample, cost ≈ N x, and the result is only as good as the verifier.
* **Serial** — one longer chain (more thinking, or refine → refine). Latency grows,
  no extra sampling.

Outcome reward scores the final answer; process reward scores every step.

In [7]:
samples = [Sample('41', 0), Sample('42', 1), Sample('40', 2)]
outcome = OutcomeVerifier({'42': 1.0, '41': 0.3, '40': 0.0})
pick = best_of_n(samples, outcome)
print(f'best_of_n (outcome verifier) -> {pick.sample.text!r} score={pick.score} tied={pick.tied}')

tied_samples = [Sample('answer A', 0), Sample('answer B', 1)]
flat = OutcomeVerifier({'answer A': 0.5, 'answer B': 0.5})
print('best_of_n without a tie-break ->', best_of_n(tied_samples, flat).sample.text, '(earliest wins, deterministic)')
tie_break = OutcomeVerifier({'answer A': 0.2, 'answer B': 0.9})
tied_pick = best_of_n(tied_samples, flat, tie_break=tie_break)
print(f'best_of_n with a tie-break    -> {tied_pick.sample.text!r} used_tie_break={tied_pick.used_tie_break}')

vote = majority_vote([Sample('yes', 0), Sample('no', 1), Sample('yes', 2), Sample('yes', 3), Sample('maybe', 4)])
print(f'majority_vote -> {vote.sample.text!r} votes={vote.votes}/{vote.total_votes} share={vote.vote_share:.1%}')

process = ProcessVerifier({'check inputs': 1.0, 'compute total': 0.2})
print('process verifier (min over steps) ->', process.score('check inputs; compute total'), '<- one bad step sinks the sample')

best_of_n (outcome verifier) -> '42' score=1.0 tied=1
best_of_n without a tie-break -> answer A (earliest wins, deterministic)
best_of_n with a tie-break    -> 'answer B' used_tie_break=True
majority_vote -> 'yes' votes=3/5 share=60.0%
process verifier (min over steps) -> 0.2 <- one bad step sinks the sample


### Parallel beats one expensive sample — when you can verify

Same hard task, same total effort spend. Best-of-8 at `medium` reaches **0.9999**
quality at the cost of a single `high` sample (0.9627), and latency stays at one
sample in both columns.

In [8]:
hard = TaskProfile('hard multi-step task', difficulty=0.9, input_tokens=800, visible_output_tokens=300)
print(f"{'effort':8s} {'n':>2s} {'thinking':>9s} {'cost':>10s} {'latency':>9s} {'quality':>8s}")
for effort in (Effort.MEDIUM, Effort.HIGH, Effort.XHIGH):
    for n in (1, 8):
        est = estimate(hard, effort, samples=n)
        print(f'{effort.value:8s} {n:2d} {est.thinking_tokens:9d} ${est.cost_usd:9.5f} {est.latency_s:8.1f}s {est.quality:8.4f}')

print()
single = estimate(hard, Effort.HIGH)
best8 = estimate(hard, Effort.HIGH, samples=8)
print(f'single high  : ${single.cost_usd:.5f}  {single.latency_s:.1f}s  q={single.quality:.4f}')
print(f'best-of-8    : ${best8.cost_usd:.5f}  {best8.latency_s:.1f}s  q={best8.quality:.4f}   (latency unchanged)')

cached = parallel_cost(Usage(800, 300, 32768), SMALL, 8, cached_input=True)
cold = parallel_cost(Usage(800, 300, 32768), SMALL, 8, cached_input=False)
print(f'8 samples, warm prefix cache = ${cached:.5f}   cold prefix cache = ${cold:.5f}')

effort    n  thinking       cost   latency  quality
medium    1      8192 $  0.01912     73.7s   0.7003
medium    8      8192 $  0.15021     73.7s   0.9999
high      1     32768 $  0.07319    278.5s   0.9627
high      8     32768 $  0.58274    278.5s   1.0000
xhigh     1     65536 $  0.14528    551.5s   0.9977
xhigh     8     65536 $  1.15946    551.5s   1.0000

single high  : $0.07319  278.5s  q=0.9627
best-of-8    : $0.58274  278.5s  q=1.0000   (latency unchanged)
8 samples, warm prefix cache = $0.58274   cold prefix cache = $0.58552


### Serial: a refinement chain shows diminishing returns

`refine` scores every version of the text, so the trace makes the curve visible.

In [9]:
rubric = {
    'draft(q)': 0.30,
    'draft(q);edited1': 0.70,
    'draft(q);edited1;edited2': 0.80,
    'draft(q);edited1;edited2;edited3': 0.85,
}
trace = refine('q', 3, verifier=OutcomeVerifier(rubric))
print('step  score  gain')
for previous, step in zip([None] + trace.steps, trace.steps):
    gain = '' if previous is None else f'{step.score - previous.score:+.2f}'
    print(f'{step.index:4d}  {step.score:.2f}  {gain:>5s}')
print('total gain:', trace.gain)

before = estimate_cost(Usage(800, 300, 4096), SMALL).total_cost
print(f'serial alternative: one call with more thinking  ${before:.5f} -> ${serial_cost(Usage(800, 300, 4096), SMALL, 8192):.5f}')

step  score  gain
   0  0.30       
   1  0.70  +0.40
   2  0.80  +0.10
   3  0.85  +0.05
total gain: 0.55
serial alternative: one call with more thinking  $0.01011 -> $0.02813


## 5. Choosing model + effort for a task profile

`choose_effort` walks the ladder and returns the **cheapest** effort that clears
the quality bar inside the budget and latency limits — and it tells you when no
rung can.

In [10]:
profiles = [
    (TaskProfile('classify a support ticket', difficulty=0.1, verifiable=True), 0.80, None),
    (TaskProfile('debug a race condition', difficulty=0.9, verifiable=True), 0.90, None),
    (TaskProfile('extract invoice fields', difficulty=0.1, verifiable=True), 0.87, 0.0015),
]
for profile, bar, budget in profiles:
    choice = choose_effort(profile, quality_bar=bar, budget_usd=budget)
    print(f"{profile.name}  (difficulty {profile.difficulty}, bar {bar}, budget {budget})")
    print(f"  -> effort={choice.effort.value} thinking={choice.estimate.thinking_tokens} "
          f"cost=${choice.estimate.cost_usd:.6f} latency={choice.estimate.latency_s:.1f}s "
          f"quality={choice.estimate.quality:.3f}")
    print(f"  met quality bar={choice.met_quality_bar}  met budget={choice.met_budget}  met latency={choice.met_latency}")
    print('  rungs considered:', ', '.join(f'{e.effort.value}({e.quality:.2f})' for e in choice.considered))
    print()

print('the whole cost/latency/quality curve for a medium task (in=800, visible=300):')
medium = TaskProfile('medium task', difficulty=0.5, input_tokens=800, visible_output_tokens=300)
print(f"{'effort':8s} {'thinking':>9s} {'cost':>10s} {'latency':>9s} {'quality':>8s}")
for effort in EFFORT_ORDER:
    est = estimate(medium, effort)
    print(f'{effort.value:8s} {est.thinking_tokens:9d} ${est.cost_usd:9.6f} {est.latency_s:8.1f}s {est.quality:8.3f}')
print(f"latency model: {BASE_LATENCY_S}s base + thinking/{THINKING_TOKENS_PER_SECOND:.0f} tok/s + visible/{VISIBLE_TOKENS_PER_SECOND:.0f} tok/s")
print(f"one-number ladder (medium difficulty): { {e.value: v for e, v in EFFORT_TO_THINKING_BUDGET.items()} }")

classify a support ticket  (difficulty 0.1, bar 0.8, budget None)
  -> effort=none thinking=0 cost=$0.000935 latency=5.4s quality=0.800
  met quality bar=True  met budget=True  met latency=True
  rungs considered: none(0.80)

debug a race condition  (difficulty 0.9, bar 0.9, budget None)
  -> effort=high thinking=32768 cost=$0.073025 latency=278.5s quality=0.963
  met quality bar=True  met budget=True  met latency=True
  rungs considered: none(0.40), minimal(0.43), low(0.50), medium(0.70), high(0.96)

extract invoice fields  (difficulty 0.1, bar 0.87, budget 0.0015)
  -> effort=max thinking=16384 cost=$0.036980 latency=141.9s quality=1.000
  met quality bar=True  met budget=False  met latency=True
  rungs considered: none(0.80), minimal(0.80), low(0.80), medium(0.87), high(0.97), xhigh(1.00), max(1.00)

the whole cost/latency/quality curve for a medium task (in=800, visible=300):
effort    thinking       cost   latency  quality
none             0 $ 0.001100      5.4s    0.600
minimal  

## 6. Prompting a reasoning model

Give the **goal**, **constraints** and **success criteria**; do **not** script the
process. Reasoning models already think, so "think step by step" is at best dead
weight. A short preamble request gets the first *visible* token out earlier.

In [11]:
prompt = build_reasoning_prompt(
    'Decide whether invoice INV-77 should be auto-approved.',
    constraints=['refunds above 500 USD need a human reviewer', 'never echo customer PII'],
    success_criteria=['returns approve/reject plus a one-line reason'],
    context=['refund policy v3 applies from 2026-01-01'],
    output_contract='JSON with keys decision and reason',
)
print(prompt)
print()
print('process-scripting phrases found:', detect_step_by_step(prompt), '<- must be empty')

scripted = 'Summarise this invoice. Think step by step. Return JSON with the total. Take a deep breath.'
cleaned, removed = detect_and_strip_step_by_step(scripted)
print('scripted prompt :', scripted)
print('removed         :', removed)
print('cleaned prompt  :', cleaned)

# Goal
Decide whether invoice INV-77 should be auto-approved.

# Constraints
- refunds above 500 USD need a human reviewer
- never echo customer PII

# Success criteria
- returns approve/reject plus a one-line reason

# Context
- refund policy v3 applies from 2026-01-01

# Output
- Return: JSON with keys decision and reason
- Open with one short sentence stating what you are about to do, then give the answer.
- Reason as deeply as the task requires; the reasoning itself does not need to be written out unless asked for.

process-scripting phrases found: [] <- must be empty
scripted prompt : Summarise this invoice. Think step by step. Return JSON with the total. Take a deep breath.
removed         : ['Think step by step.', 'Take a deep breath.']
cleaned prompt  : Summarise this invoice. Return JSON with the total.


## 7. Planner/worker routing

A reasoning model plans, decomposes and audits; a fast cheap model executes.
Hard or high-value turns go to the reasoning model — the topic 19c auto-tier idea
inside one loop.

In [12]:
reasoning_large = ModelSpec('reasoning-large', 'reasoning', PRICE_TABLE['reasoning-large'], 800, 400)
fast_small = ModelSpec('fast-small', 'fast', PRICE_TABLE['fast-small'], 400, 200)

subtasks = [
    SubTask('parse the invoice header', difficulty=0.2, value='low'),
    SubTask('reconcile totals across documents', difficulty=0.85, value='medium'),
    SubTask('decide the refund outcome', difficulty=0.4, value='high'),
]
plan = planner_worker_split(reasoning_large, fast_small, subtasks)
for assignment in plan.assignments:
    print(f'{assignment.subtask:38s} -> {assignment.model:16s} ({assignment.reason})')
print('summary:', plan.summary())

specs = {reasoning_large.name: reasoning_large, fast_small.name: fast_small}
mixed_cost = sum(specs[a.model].call_cost(0) for a in plan.assignments)
all_reasoning = len(plan.assignments) * reasoning_large.call_cost(0)
print(f'plan cost ${mixed_cost:.5f} vs all-reasoning ${all_reasoning:.5f}  ({(1 - mixed_cost / all_reasoning) * 100:.0f}% cheaper)')

print()
print('single-turn routing:')
for subtask in [SubTask('look up order status', 0.1), SubTask('design the migration plan', 0.05, needs_planning=True)]:
    routed = route_turn(subtask, fast_small, reasoning_large)
    print(f'  {subtask.name:32s} -> {routed.model}  ({routed.reason})')

<plan>                                 -> reasoning-large  (decompose the task into sub-tasks (planning turn))
parse the invoice header               -> fast-small       (easy turn (difficulty 0.20) - fast model executes)
reconcile totals across documents      -> reasoning-large  (hard turn (difficulty 0.85 >= 0.60))
decide the refund outcome              -> reasoning-large  (high-value turn)
<audit>                                -> reasoning-large  (audit the assembled result before it ships)
summary: {'plan_model_calls': 4, 'worker_model_calls': 1, 'total_calls': 5}
plan cost $0.02418 vs all-reasoning $0.03000  (19% cheaper)

single-turn routing:
  look up order status             -> fast-small  (easy turn (difficulty 0.10) - fast model executes)
  design the migration plan        -> reasoning-large  (needs planning/decomposition)


## 8. Takeaways

- Thinking tokens are hidden, occupy the window, and are **billed as output** —
  on a 4,096-token thinking call they can be ~89% of the bill.
- `max_output_tokens` caps thinking + output: run out while thinking and you get
  `incomplete / max_output_tokens`, no visible text, and a bill.
- Extra thinking earns its keep on hard, **verifiable** work; the curve flattens
  fast (`xhigh -> max` bought +0.004 quality for +$0.072 in the table above).
- Parallel TTC (best-of-N / self-consistency) costs ~N x at one sample's latency
  and is only as good as the verifier; serial TTC (longer chain) trades latency
  for quality with no verifier at all.
- Prompt with goals, constraints and success criteria — the model already thinks.
- Route plan/audit turns to a reasoning model and execution turns to a fast one;
  that is the same difficulty routing as topic 19c's auto-tiers.